# 🏛️ LutherICPU — 100% Native Custom 3D Neural Reconstructor
**Train your own custom from-scratch 3D Neural Radiance & Surface Simulation model.**

This notebook runs **100% native LutherICPU code** from your repository without third-party models.

---
### 🚀 How to Run:
1. In the top menu, go to **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** (or High-RAM CPU).
2. Run all cells below in order.
3. Upload your `images.zip` containing your source photographs.
4. When training finishes, your custom `luther_simulation_bundle.zip` will automatically download.
5. On your computer, run `python simulate.py -b luther_simulation_bundle.zip` to launch the 60 FPS interactive 3D simulation!

In [ ]:
# Step 1: Clone your OWN native LutherICPU Repository (Zero Third-Party Repos)
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq colmap
!git clone https://github.com/AryanParikh31/LutherICPU.git /content/LutherICPU
%cd /content/LutherICPU
!pip install -q -r requirements.txt
!pip install -q plyfile tqdm scipy trimesh open3d onnxruntime

In [ ]:
# Step 2: Upload your Photographs (Zip archive)
import os, zipfile, shutil
from google.colab import files

workspace_dir = "/content/dataset"
if os.path.exists(workspace_dir):
    shutil.rmtree(workspace_dir)
os.makedirs(os.path.join(workspace_dir, "images"), exist_ok=True)

print("[*] Please upload your images zip file (e.g. drjohnson_images.zip):")
uploaded = files.upload()
zip_filename = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(os.path.join(workspace_dir, "input"))

# Flatten images into /content/dataset/images
input_path = os.path.join(workspace_dir, "input")
valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".PNG", ".JPEG")
all_files = []
for root, dirs, fnames in os.walk(input_path):
    for f in fnames:
        if f.endswith(valid_exts):
            all_files.append(os.path.join(root, f))

target_input = os.path.join(workspace_dir, "images")
for f in all_files:
    shutil.copy2(f, os.path.join(target_input, os.path.basename(f)))

print(f"[SUCCESS] Loaded {len(all_files)} photographs into native LutherICPU workspace.")

In [ ]:
# Step 3: Run Native Luther Structure-from-Motion & Camera Calibration
%cd /content/LutherICPU
!python -m luther_core.ingestion --images /content/dataset/images --output /content/scene_output

In [ ]:
# Step 4: Train Native Luther 3D Neural Radiance & Surface Model (30,000 Iterations)
%cd /content/LutherICPU
from luther_pipeline.simulation_model import LutherICPU

print("[*] Training Native LutherICPU Photometric 3D Model from Scratch...")
model = LutherICPU(output_dir="/content/scene_output/model", max_ram_gb=14.0)
results = model.simulate(
    images_path="/content/dataset/images",
    iterations=30000,
    scene_name="drjohnson",
    render_simulation=True
)
print("[SUCCESS] Native LutherICPU 3D Model Training Complete!")
print(results)

In [ ]:
# Step 5: Package Native Simulation Bundle & Download Automatically
import os, zipfile
from google.colab import files

out_model_dir = "/content/scene_output/model"
bundle_zip = "/content/luther_simulation_bundle.zip"

print("[*] Packaging Native Luther 3D Simulation Assets...")
with zipfile.ZipFile(bundle_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, fnames in os.walk(out_model_dir):
        for f in fnames:
            full_p = os.path.join(root, f)
            rel_p = os.path.relpath(full_p, out_model_dir)
            zipf.write(full_p, rel_p)

print(f"[SUCCESS] Bundle created: {os.path.getsize(bundle_zip) / (1024*1024):.1f} MB")
print("[*] Initiating automatic download to your local PC...")
files.download(bundle_zip)